In [1]:
import scipy
import logging
import datetime
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [4]:
SUBJECT_IDS = [f'f{i:02d}' for i in range(1, 7)] + [f'f{i:02d}' for i in range(8, 14)] + [f'f{i:02d}' for i in range(15, 19)]
TEST_SUBJECTS = 5
SAMPLING_RATE = 4
SAMPLING_RATES = {
    'ACC': 32,
    'BVP': 64,
    'EDA': 4,
    'TEMP': 4,
    'label': 4,
}

In [5]:
# create a vector from the data frame (signal imported by pandas)
def create_df_array(dataframe):
    matrix_df=dataframe.values
    # returns 2-d matrix
    matrix = np.array(matrix_df)
    array_df = matrix.flatten()# Convert matrix into an array
    return array_df

# convert UTC arrays to arrays in seconds relative to 0 (record beginning)
def time_abs_(UTC_array):
    new_array=[]
    for utc in UTC_array:
        time=(datetime.datetime.strptime(utc,'%Y-%m-%d %H:%M:%S')-datetime.datetime.strptime(UTC_array[0], '%Y-%m-%d %H:%M:%S')).total_seconds()
        new_array.append(int(time))
    return new_array

In [6]:
def read_signals(main_folder):
    signal_dict = {}

    # Get a list of subfolders in the main folder
    subfolders = [p for p in Path(main_folder).iterdir() if p.is_dir()]

    utc_start_dict={}
    for folder_path in subfolders:
            csv_path = folder_path / 'EDA.csv'
            df=pd.read_csv(csv_path)
            utc_start_dict[folder_path.name]= df.columns.tolist()

    # Iterate over the subfolders
    for folder_path in subfolders:
        folder_name = folder_path.name

        # Initialize a dictionary for the signals in the current subfolder
        signals = {}
        
        # Define the list of desired file names
        desired_files = ['EDA.csv', 'BVP.csv', 'TEMP.csv', 'tags.csv', 'ACC.csv']
   
        # Iterate over the files in the subfolder
        for file_path in folder_path.iterdir():
            file_name = file_path.name

            # Check if it's a CSV file and if it is in the desired files list
            if file_name.endswith('.csv') and file_name in desired_files:
                # Read the CSV file and store the signal data
                if file_name == 'tags.csv':
                    try:
                        df = pd.read_csv(file_path, header=None)
                        tags_vector = create_df_array(df)
                        tags_UTC_vector = np.insert(tags_vector, 0, utc_start_dict[folder_name])
                        signal_array = time_abs_(tags_UTC_vector)
                    except pd.errors.EmptyDataError:
                        signal_array=[]
                else:
                    df = pd.read_csv(file_path)
                    df.drop([0], axis=0, inplace=True)
                    signal_array = df.values
                
                signal_name = file_name.split('.')[0]
                signals[signal_name] = signal_array

        # Store the signals of the current subfolder in the main dictionary
        signal_dict[folder_name] = signals

    return signal_dict

In [7]:
def eda_process(
    eda_signal, sampling_rate=1000, method="neurokit", report=None, **kwargs
):
    # Sanitize input
    eda_signal = nk.signal.signal_sanitize(eda_signal)
    methods = nk.eda.eda_methods.eda_methods(sampling_rate=sampling_rate, method=method, **kwargs)

    # Preprocess
    # Clean signal
    eda_cleaned = eda_signal
    if methods["method_phasic"] is None or methods["method_phasic"].lower() == "none":
        eda_decomposed = pd.DataFrame({"EDA_Phasic": eda_cleaned})
    else:
        eda_decomposed = nk.eda_phasic(
            eda_cleaned,
            sampling_rate=sampling_rate,
            method=methods["method_phasic"],
            **methods["kwargs_phasic"],
        )

    # Find peaks
    peak_signal, info = nk.eda_peaks(
        eda_decomposed["EDA_Phasic"].values,
        sampling_rate=sampling_rate,
        method=methods["method_peaks"],
        amplitude_min=0.1,
        **methods["kwargs_peaks"],
    )
    info["sampling_rate"] = sampling_rate  # Add sampling rate in dict info

    # Store
    signals = pd.DataFrame({"EDA_Raw": eda_signal, "EDA_Clean": eda_cleaned})

    signals = pd.concat([signals, eda_decomposed, peak_signal], axis=1)

    return signals, info

In [8]:
def ppg_process(
    ppg_signal, sampling_rate=1000, method="elgendi", method_quality="templatematch", report=None, **kwargs
):
    # Sanitize input
    ppg_signal = nk.misc.as_vector(ppg_signal)
    methods = nk.ppg.ppg_methods(sampling_rate=sampling_rate, method=method, method_quality=method_quality, **kwargs)

    # Clean signal
    ppg_cleaned = ppg_signal

    # Find peaks
    peaks_signal, info = nk.ppg_peaks(
        ppg_cleaned,
        sampling_rate=sampling_rate,
        method="bishop",
    )

    info["sampling_rate"] = sampling_rate  # Add sampling rate in dict info

    # Rate computation
    rate = nk.signal.signal_rate(
        info["PPG_Peaks"], sampling_rate=sampling_rate, desired_length=len(ppg_cleaned)
    )

    # Assess signal quality
    quality = nk.ppg_quality(
        ppg_cleaned,
        peaks=info["PPG_Peaks"],
        sampling_rate=sampling_rate,
        method=methods["method_quality"],
        **methods["kwargs_quality"]
    )

    # Prepare output
    signals = pd.DataFrame(
        {
            "PPG_Raw": ppg_signal,
            "PPG_Clean": ppg_cleaned,
            "PPG_Rate": rate,
            "PPG_Quality": quality,
            "PPG_Peaks": peaks_signal["PPG_Peaks"].values,
        }
    )

    return signals, info


In [9]:
def process_data(data: dict, subject_id: str, sampling_rate: int = 4):
    label = -np.ones(len(data['EDA']))
    if subject_id.startswith('S'):
        label[data['tags'][1]:data['tags'][2]] = 0  # baseline
        label[data['tags'][3]:data['tags'][4]] = 1  # stroop
        #label[data['tags'][4]:data['tags'][5]] = 0  # first rest
        label[data['tags'][5]:data['tags'][6]] = 1  # tmct
        #label[data['tags'][6]:data['tags'][7]] = 0  # second rest
        label[data['tags'][7]:data['tags'][8]] = 1  # real opinion
        label[data['tags'][9]:data['tags'][10]] = 1  # opposite opinion
        label[data['tags'][11]:data['tags'][12]] = 1  # subtract test
    else:
        label[:data['tags'][1]] = 0  # baseline
        label[data['tags'][2]:data['tags'][3]] = 1  # tmct
        #label[data['tags'][3]:data['tags'][4]] = 0  # first rest
        label[data['tags'][4]:data['tags'][5]] = 1  # real opinion
        label[data['tags'][6]:data['tags'][7]] = 1  # opposite opinion
        #label[data['tags'][7]:data['tags'][8]] = 0  # second rest
        label[data['tags'][8]:data['tags'][9]] = 1  # subtract test
    del data['tags']
    
    if SAMPLING_RATES['label'] > sampling_rate:
        label = label[::SAMPLING_RATES['label'] // sampling_rate]
    elif SAMPLING_RATES['label'] < sampling_rate:
        time = np.round(np.arange(0, len(label)) / SAMPLING_RATES['label'], 3)
        f = scipy.interpolate.interp1d(time, label, kind='nearest', fill_value='extrapolate')
        time_new = np.arange(0, time[-1], 1 / sampling_rate)
        label = f(time_new)
    
    for key in data.keys():
        if key == 'ACC':
            temp = []
            for i in range(data[key].shape[1]):
                temp.append(nk.signal_resample(data[key][:, i], sampling_rate=SAMPLING_RATES[key], desired_sampling_rate=SAMPLING_RATE))
            temp.append(np.linalg.norm(np.array(temp), axis=0))
            data['ACC'] = np.stack(temp, axis=1)
        else:
            resampled = nk.signal_resample(data[key][:, 0], sampling_rate=SAMPLING_RATES[key], desired_sampling_rate=SAMPLING_RATE)
            data[key] = resampled

    bvp, _ = ppg_process(data['BVP'], sampling_rate=SAMPLING_RATE)
    eda, _ = eda_process(data['EDA'], sampling_rate=SAMPLING_RATE)
    
    acc = pd.DataFrame(data['ACC'], columns=['ACC_x', 'ACC_y', 'ACC_z', 'ACC_net'])
    bvp = bvp.drop(columns=['PPG_Raw'])
    eda = eda.drop(columns=['EDA_Raw'])
    temp = pd.DataFrame(data['TEMP'], columns=['TEMP'])

    label = pd.DataFrame(label, columns=['label'])
    
    # Merge all dataframes on their time indices
    df = pd.concat([acc, bvp, eda, temp, label], axis=1, join='outer')
    
    # Filter labels to include only 0 (baseline) and 1 (stress)
    df = df[df['label'].isin([0, 1])]
    
    # Reset index to have a clean integer index
    df.reset_index(drop=True, inplace=True)
    
    df = df.assign(
        subject_id=subject_id
    )

    return df

In [10]:
def process_all_data(signal_data: dict, sampling_rate: int = 4):
    all_data = {}
    for subject_id in tqdm(SUBJECT_IDS):
        df_subject = process_data(signal_data[subject_id], subject_id, sampling_rate)
        all_data[subject_id] = df_subject
    
    logger.info('All data processed')

    return all_data

In [21]:
def data_split(all_data, test_subjects: int):
    all_data_list = list(all_data.values())
    n_subjects = len(all_data)
    n_test = test_subjects
    
    n_train = n_subjects - n_test

    df_train = pd.concat(all_data_list[:n_train], ignore_index=True, axis=0)
    df_test = pd.concat(all_data_list[n_train:], ignore_index=True, axis=0)

    df_train_with_anomaly = df_train.reset_index(drop=True)
    df_train = df_train[df_train['label'] == 0].reset_index(drop=True) # Use only non-stress data for training 
    df_test = df_test.reset_index(drop=True) # Use all data for testing
    
    logger.info('Data split into train and test sets')
    logger.info('Train data shape: %s', df_train.shape)
    logger.info('Test data shape: %s', df_test.shape)

    return df_train, df_train_with_anomaly, df_test

In [12]:
data_path = Path('physionet.org') / "files" / "wearable-device-dataset" / "1.0.1" / "Wearable_Dataset" / "STRESS"
signal_data = read_signals(data_path) # Returns three dictionaries with subjects info: raw signals (signal_data), temporal data ready to graph (time_data) and sample frequency for escha signal(fs_dict).

In [13]:
all_data = process_all_data(signal_data, sampling_rate=SAMPLING_RATE)

100%|██████████| 16/16 [04:38<00:00, 17.43s/it]
INFO:__main__:All data processed


In [22]:
df_train, df_train_with_anomaly, df_test = data_split(all_data, TEST_SUBJECTS)

INFO:__main__:Data split into train and test sets
INFO:__main__:Train data shape: (10245, 21)
INFO:__main__:Test data shape: (7512, 21)


In [23]:
df_train.ffill(inplace=True)
df_train_with_anomaly.ffill(inplace=True)
df_test.ffill(inplace=True)

In [24]:
Path('data_2').mkdir(parents=True, exist_ok=True)
df_train.to_csv('data_2/train.csv', index=False)
df_train_with_anomaly.to_csv('data_2/train_with_anomaly.csv', index=False)
df_test.to_csv('data_2/test.csv', index=False)

In [25]:
df_train.label.value_counts()

label
0.0    10245
Name: count, dtype: int64

In [26]:
df_train_with_anomaly.label.value_counts()

label
0.0    10245
1.0     4449
Name: count, dtype: int64

In [27]:
df_test.label.value_counts()

label
0.0    5668
1.0    1844
Name: count, dtype: int64

In [28]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10245 entries, 0 to 10244
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ACC_x             10245 non-null  float64
 1   ACC_y             10245 non-null  float64
 2   ACC_z             10245 non-null  float64
 3   ACC_net           10245 non-null  float64
 4   PPG_Clean         10245 non-null  float64
 5   PPG_Rate          10245 non-null  float64
 6   PPG_Quality       10245 non-null  float64
 7   PPG_Peaks         10245 non-null  float64
 8   EDA_Clean         10245 non-null  float64
 9   EDA_Tonic         10245 non-null  float64
 10  EDA_Phasic        10245 non-null  float64
 11  SCR_Onsets        10245 non-null  float64
 12  SCR_Peaks         10245 non-null  float64
 13  SCR_Height        10245 non-null  float64
 14  SCR_Amplitude     10245 non-null  float64
 15  SCR_RiseTime      10245 non-null  float64
 16  SCR_Recovery      10245 non-null  float6